# 프롬프트 버전 관리

In [4]:
import vertexai
from vertexai import types
from google import genai
from google.genai import types as genai_types

PROJECT_ID = "byounghwa-go"
LOCATION = "us-central1"

# Vertex AI SDK 클라이언트 초기화
vertex_client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

# Google GenAI SDK 클라이언트 초기화
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

# 프롬프트 객체 생성
prompt = types.Prompt(
    prompt_data=types.PromptData(
        model="gemini-3-flash-preview",
        contents=[
            genai_types.Content(
                role="user",
                parts=[
                    genai_types.Part(text="안녕하세요.")
                ]
            )
        ],
        system_instruction=genai_types.Content(
            parts=[
                genai_types.Part(
                    text="""당신은 고객 문의에 친절하고 정확하게 답변하는 전문 상담원입니다.
모든 답변은 공손한 어투로 작성하며, 고객이 이해하기 쉽도록 명확하게 설명합니다."""
                )
            ]
        )
    ),
)

prompt_resource = vertex_client.prompts.create(prompt=prompt)   # 장시간 소요됨!!
PROMPT_ID = prompt_resource.prompt_id
print(f"프롬프트 리소스 생성 완료. 프롬프트 ID: {PROMPT_ID}")

프롬프트 리소스 생성 완료. 프롬프트 ID: 2245281358999977984


In [5]:
# 프롬프트 버전 저장
prompt_version_resource = vertex_client.prompts.create_version(
    prompt=prompt,
    prompt_id=PROMPT_ID
)
print(f"프롬프트 버전 저장 완료. 프롬프트 버전 ID: {prompt_version_resource.version_id}")

프롬프트 버전 저장 완료. 프롬프트 버전 ID: 1


In [6]:
# 새로운 프롬프트 정의
prompt_v2 = types.Prompt(
    prompt_data=types.PromptData(
        model="gemini-3-flash-preview",
        contents=[
            genai_types.Content(
                parts=[genai_types.Part(text="안녕하세요.")]
            )
        ],
        system_instruction=genai_types.Content(
            parts=[
                genai_types.Part(
                    text="""당신은 신속하고 간결하게 핵심만 전달하는 상담원입니다.
불필요한 설명은 생략하고, 고객이 요청한 내용만 명확히 답변합니다."""
                )
            ]
        )
    )
)

In [9]:
# 버전 2로 저장
prompt_version_v2 = vertex_client.prompts.create_version(
    prompt=prompt_v2,
    prompt_id=PROMPT_ID
)
print(f"프롬프트 버전 2 저장 완료. 프롬프트 버전 ID: {prompt_version_v2.version_id}")

프롬프트 버전 2 저장 완료. 프롬프트 버전 ID: 3


In [10]:
# 프롬프트 조회 

# 사용 모델 및 프롬프트 ID 가져오기
prompt_refs = list(vertex_client.prompts.list())
print(f"사용 모델 및 프롬프트 ID:\n{prompt_refs}")

# 프롬프트 가져오기
get_prompt = vertex_client.prompts.get(prompt_id=PROMPT_ID)
print(f"프롬프트:\n{get_prompt}")

# 프롬프트 버전 확인
prompt_versions_metadata = list(vertex_client.prompts.list_versions(prompt_id=PROMPT_ID))
print(f"프롬프트 버전:\n{prompt_versions_metadata}")

사용 모델 및 프롬프트 ID:
[PromptRef(
  model='gemini-3-flash-preview',
  prompt_id='2245281358999977984'
)]
프롬프트:
prompt_data=SchemaPromptSpecPromptMessage(
  contents=[
    Content(
      parts=[
        Part(
          text='안녕하세요.'
        ),
      ]
    ),
  ],
  model='gemini-3-flash-preview',
  system_instruction=Content(
    parts=[
      Part(
        text="""당신은 신속하고 간결하게 핵심만 전달하는 상담원입니다.
불필요한 설명은 생략하고, 고객이 요청한 내용만 명확히 답변합니다."""
      ),
    ]
  )
)
프롬프트 버전:
[PromptVersionRef(
  model='gemini-3-flash-preview',
  prompt_id='2245281358999977984',
  version_id='1'
), PromptVersionRef(
  model='gemini-3-flash-preview',
  prompt_id='2245281358999977984',
  version_id='2'
), PromptVersionRef(
  model='gemini-3-flash-preview',
  prompt_id='2245281358999977984',
  version_id='3'
)]


In [11]:
# 프롬프트 롤백 
restored_prompt = vertex_client.prompts.restore_version(
    prompt_id=PROMPT_ID,
    version_id="1"
)
print(f"버전 1로 롤백 완료. 현재 프롬프트 버전: {restored_prompt.version_id}")

버전 1로 롤백 완료. 현재 프롬프트 버전: 1


In [12]:
# Gemini에 프롬프트 적용
# 최신 버전의 프롬프트 가져오기
retrieved_prompt = vertex_client.prompts.get(prompt_id=PROMPT_ID)
print(f"로드된 모델: {retrieved_prompt.prompt_data.model}")
print(f"시스템 지시문: {retrieved_prompt.prompt_data.system_instruction}")

# GCP 콘솔의 Vertex AI --> AI Studio 의 "최신 소식" 에서 저장된 프롬프트를 확인할수 있다

로드된 모델: gemini-3-flash-preview
시스템 지시문: parts=[Part(
  text="""당신은 고객 문의에 친절하고 정확하게 답변하는 전문 상담원입니다.
모든 답변은 공손한 어투로 작성하며, 고객이 이해하기 쉽도록 명확하게 설명합니다."""
)] role=None


In [14]:
# 롤백된 프롬프트로 Gemini 모델 호출하기
PROJECT_ID = "byounghwa-go"
LOCATION = "global"

# Google GenAI SDK 클라이언트 초기화
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

response = genai_client.models.generate_content(
    model=retrieved_prompt.prompt_data.model,
    contents="제품 반품 절차를 알려주세요.",
    config=genai_types.GenerateContentConfig(
        system_instruction=retrieved_prompt.prompt_data.system_instruction
    )
)

print(response.text)

안녕하세요, 고객님! 제품 반품에 대해 문의해 주셔서 감사합니다.
구매하신 제품의 반품 절차를 알기 쉽게 단계별로 안내해 드리겠습니다.

---

### **[제품 반품 절차 안내]**

**1. 반품 신청**
*   **온라인 접수:** 저희 홈페이지 또는 앱의 **[마이페이지 > 주문/배송 조회]** 메뉴에서 반품을 원하시는 주문 건을 선택하신 후 '반품 신청' 버튼을 눌러주세요.
*   **고객센터 접수:** 온라인 신청이 어려우신 경우, 고객센터(전화번호: 0000-0000)로 연락 주시면 상담원이 직접 접수를 도와드립니다.

**2. 제품 재포장**
*   제품을 받으셨을 때와 동일한 상태로 포장해 주세요.
*   **주의사항:** 제품 박스, 구성품(설명서, 케이블 등), 증정된 사은품이 모두 포함되어야 반품이 가능합니다.

**3. 제품 회수(택배)**
*   반품 신청 시 '자동 수거'를 선택하시면, 영업일 기준 1~3일 이내에 택배 기사님이 고객님의 주소지로 방문하여 제품을 수거합니다.
*   직접 택배를 보내실 경우, 지정된 반송처 주소로 보내주시기 바랍니다. (단, 타 택배사 이용 시 추가 비용이 발생할 수 있습니다.)

**4. 검수 및 환불**
*   반품된 제품이 저희 물류 센터에 도착하면 제품의 상태를 확인하는 검수 과정을 거칩니다.
*   검수가 완료되면 영업일 기준 3~5일 이내에 결제하셨던 수단(신용카드 취소, 계좌 입금 등)으로 환불 처리가 진행됩니다.

---

### **※ 유의사항을 확인해 주세요**
*   **반품 가능 기간:** 제품 수령 후 **7일 이내**에 신청해 주셔야 합니다.
*   **반품 배송비:** 제품 하자가 아닌 고객님의 단순 변심으로 인한 반품의 경우, 왕복 배송비가 고객님 부담으로 발생할 수 있습니다.
*   **반품 불가 사유:** 제품을 사용했거나 포장이 훼손되어 상품 가치가 상실된 경우에는 반품이 어려울 수 있으니 주의 부탁드립니다.

안내해 드린 내용 중 더 궁금하신 점이 있거나 도움이 필

In [15]:
# 프롬프트 삭제 

# 특정 버전만 삭제
vertex_client.prompts.delete_version(
    prompt_id=PROMPT_ID,
    version_id="2"
)
print("버전 2가 삭제되었습니다.")

버전 2가 삭제되었습니다.


In [16]:
# 프롬프트와 모든 버전 삭제
vertex_client.prompts.delete(prompt_id=PROMPT_ID)